<a href="https://colab.research.google.com/github/almendraapolaya/DI_Bootcamp_a/blob/main/Week_7/Day_4/Daily_challenge%20/Daily_challenge_day4_w7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Daily Challenge :
===
 How to Finetune LLMs with LoRA


**Task**:

- Install necessary libraries (PEFT, datasets).
- Load a pre-trained language model (bigscience/bloomz-560m) and its tokenizer.
- Load the dataset and preprocess it for the model.
- Configure LoRA using LoraConfig.
- Apply LoRA to the pre-trained model using get_peft_model.
- Set up training arguments using TrainingArguments.
- Initialize and train the model using Trainer.
- Save the fine-tuned LoRA model.
- Load the saved LoRA model for inference using PeftModel.from_pretrained.
- Generate text using the fine-tuned model and the tokenizer.

**Install necessary libraries (PEFT, datasets):**

In [ ]:
!pip install -U -q peft datasets transformers accelerate

**Load a pre-trained language model (bigscience/bloomz-560m) and its tokenizer:**

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "bigscience/bloomz-560m"

tokenizer = AutoTokenizer.from_pretrained(model_name)

foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

print("Step 2 Complete: Model and Tokenizer loaded.")

**Load the dataset and preprocess it for the model:**

In [ ]:
from datasets import load_dataset

data = load_dataset("Abirate/english_quotes", split="train")

data = data.train_test_split(test_size=0.1, seed=42)["test"]

data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)

train_sample = data.select(range(50))

print("Step 3 Complete: 10% sample loaded and tokenized.")

**Configure LoRA using LoraConfig:**

In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=8,
    lora_alpha=32,
    target_modules=["query_key_value"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

print("Step 4 Complete: LoRA Configuration defined.")

**Apply LoRA to the pre-trained model using get_peft_model:**

In [ ]:
!pip install -U torchao -q

In [ ]:
from peft import get_peft_model

peft_model = get_peft_model(foundation_model, lora_config)

peft_model.print_trainable_parameters()

print("Step 5 Complete: LoRA adapters applied to the model.")

**Set up training arguments using TrainingArguments:**

In [ ]:
import os
from transformers import TrainingArguments

output_directory = os.path.join("cache", "peft_outputs")

training_args = TrainingArguments(
    output_dir=output_directory,
    auto_find_batch_size=True,
    learning_rate=3e-2,
    num_train_epochs=1,
    logging_steps=10,
    report_to="none",
    use_cpu=False
)

print("Step 6 Complete: Training arguments set.")

**Initialize and train the model using Trainer:**

In [ ]:
import transformers
from transformers import Trainer

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
)

trainer.train()

print("Step 7 Complete: Training finished!")

**Save the fine-tuned LoRA model:**

In [ ]:
import time

time_now = int(time.time())
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")
trainer.model.save_pretrained(peft_model_path)

print(f"Folder created at: {peft_model_path}")



**Load the saved LoRA model for inference using PeftModel.from_pretrained:**

In [ ]:
from peft import PeftModel

inference_model = PeftModel.from_pretrained(
    foundation_model,
    peft_model_path,
    is_trainable=False
)

print("Step 9 Complete: LoRA model loaded and ready for inference.")

**Generate text using the fine-tuned model and the tokenizer:**

In [ ]:
from peft import PeftModel
import torch

inference_model = PeftModel.from_pretrained(foundation_model, peft_model_path).to("cuda")
inference_model.eval()

prompt = "Be the change that you wish to "
inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

print("Generating with Beam Search...")
with torch.no_grad():
    outputs = inference_model.generate(
        **inputs,
        max_new_tokens=40,
        do_sample=False,
        num_beams=5,
        no_repeat_ngram_size=2,
        pad_token_id=tokenizer.eos_token_id
    )

print("\n--- GENERATED QUOTE ---")
print(tokenizer.batch_decode(outputs, skip_special_tokens=True)[0])